# Serverless job lifecycle

Runs one GPU job on Nebius Serverless AI Jobs and walks through its lifecycle: describe it, dry-run it, create it, check its status, wait for it, read its logs, reload it by id, and cancel it.

`examples/quickstarts/job_quickstart.py` is the shorter version: create and wait only.

**Before you start**

- `uv sync` in the repo root, then start Jupyter from the same environment, for example `uv run --with jupyter jupyter lab`.
- A Nebius CLI profile (`nebius profile create`). The active profile supplies the project. The region comes from the profile or the project.
- Quota for one on-demand H100 in `eu-north1`. Change `gpu=` for another region; machine selection is in `docs/guides/machines.md`.

Jupyter already runs an event loop, so every call that talks to Nebius uses the awaited twin (`create_async`, `wait_async`, …). A synchronous `create()` in this kernel raises `ValidationError` and names the twin to use.

In [1]:
import nebius_serverless as ns

ns.diagnostics()

{'version': '0.1.0',
 'python': '3.12.12',
 'platform': 'macOS-27.2.1-arm64-arm-64bit',
 'presentation': 'plain'}

## Check the account

`whoami_async` makes one read-only call and resolves the project and region exactly as `create` will. Nothing is created. `quiet=True` skips the summary line, which names the account.

In [2]:
who = await ns.whoami_async(quiet=True)

who["account"]["kind"], who["auth"]["source"], who["region"]

('user', 'profile', 'eu-north1')

## 1. Describe the job

A `JobSpec` is data: what to run. Constructing it does not contact Nebius. `ns.Job(spec)` is the job built from it, and nothing is created until `create`.

The command is a list of strings. To run a shell pipeline, pass it to `bash -lc`. `env` values are sent on create and never shown in dry-run output or error text.

In [3]:
spec = ns.JobSpec(
    command=[
        "bash",
        "-lc",
        "nvidia-smi --query-gpu=name,memory.total --format=csv"
        ' && echo "hello from $GREETING"',
    ],
    image="nvidia/cuda:13.1.1-runtime-ubuntu24.04",
    gpu="h100",
    timeout="1h",
    env={"GREETING": "a notebook"},
    name="notebook-lifecycle",
)

{
    "name": spec.name,
    "command": list(spec.command),
    "image": spec.image,
    "gpu": spec.gpu,
    "pricing": spec.pricing,
    "timeout": spec.timeout,
}

{'name': 'notebook-lifecycle',
 'command': ['bash',
  '-lc',
  'nvidia-smi --query-gpu=name,memory.total --format=csv && echo "hello from $GREETING"'],
 'image': 'nvidia/cuda:13.1.1-runtime-ubuntu24.04',
 'gpu': 'h100',
 'pricing': 'on-demand',
 'timeout_seconds': 3600}

## 2. Dry run

Resolves the project and region and returns the create request. No job is created. `gpu` is sent as a selector; Nebius picks the platform and preset at create, and `display()` shows them after a fetch.

In [4]:
job = ns.Job(spec)
job.dry_run()

{'image': 'nvidia/cuda:13.1.1-runtime-ubuntu24.04',
 'command': ['bash',
  '-lc',
  'nvidia-smi --query-gpu=name,memory.total --format=csv && echo "hello from $GREETING"'],
 'args': '-lc \'nvidia-smi --query-gpu=name,memory.total --format=csv && echo "hello from $GREETING"\'',
 'project': 'project-e00…',
 'region': 'eu-north1',
 'gpu': 'h100',
 'timeout': 3600,
 'pricing': 'on-demand',
 'env': ['GREETING'],
 'name': 'notebook-lifecycle',
 'disk': '250Gi',
 'shm': '16Gi'}

## 3. Create

Returns a `Job` as soon as Nebius accepts the job and its id is known. It does not wait for the container. **This cell starts a billable job.**

Save `job.id`. It is the only handle you need to find the job again from another notebook or terminal.

In [5]:
await job.create_async()

job_id = job.id
job_id, job.status

Submitted job aijob-e00… notebook-lifecycle


('aijob-e00…', None)

## 4. Check status without blocking

`get_async` fetches the job once and stores the result on `job.record` and `job.status`. Re-run this cell to watch the state change. A failed container is data here, not an exception.

In [6]:
await job.get_async()
job.status

aijob-e00… PROVISIONING


'PROVISIONING'

## 5. Wait for a terminal state

Polls about every 10 seconds until the state is `COMPLETED`, `FAILED`, `CANCELLED`, `ERROR`, or `DELETING`.

`timeout=` here limits only the local wait. When it expires, `WaitTimeout` is raised and the job keeps running in the cloud. Re-run the cell to keep waiting.

In [7]:
try:
    await job.wait_async(timeout=30 * 60)
except ns.WaitTimeout as exc:
    print(f"Stopped waiting; job {exc.resource_id} is still {job.status}.")
    print("Re-run this cell to keep waiting, or cancel it in step 8.")
else:
    job.display()

aijob-e00… PROVISIONING · 0.1s


aijob-e00… STARTING · 131.8s


aijob-e00… IMAGE_PULLING · 172.3s


✓ aijob-e00… COMPLETED · 192.5s


aijob-e00…  notebook-lifecycle


  COMPLETED


... (10 outputs left out)

  finished  2026-11-19 22:04:24Z


  queued    3m 13s


  ran       0s


  instance  computeinstance-e00…  10.0.0.17  COMPLETED


## 6. Read the logs

This client does not stream container logs. The Nebius CLI does. IPython substitutes `{job_id}` into the shell command. You should see the GPU name and `hello from a notebook`.

In [8]:
!nebius ai job logs {job_id}

Provisioning resources for the workload
Workload initialization started
Starting workload container
name, memory.total [MiB]
NVIDIA H100 80GB HBM3, 81559 MiB
hello from a notebook


## 7. Reload by id

`jobs.get_async` starts from an id string: after a kernel restart, in another notebook, or for a job created from the CLI. It fetches once and never creates a job. `record` is the raw job JSON.

In [9]:
loaded = await ns.jobs.get_async(job_id)

print(loaded.id, loaded.name, loaded.status)
sorted(loaded.record)

aijob-e00… COMPLETED


aijob-e00… notebook-lifecycle COMPLETED


['metadata', 'spec', 'status']

## 8. Cancel

Interrupting a cell, a local `WaitTimeout`, and restarting the kernel all leave the job running. Only `cancel_async` stops it. It returns once the request is accepted; a later `get_async` shows `CANCELLED`.

The cell skips the request when the job has already finished.

In [10]:
TERMINAL = {"COMPLETED", "FAILED", "CANCELLED", "ERROR", "DELETING"}

await job.get_async(quiet=True)

if job.status in TERMINAL:
    print(f"Job {job.id} already finished: {job.status}")
else:
    await job.cancel_async()
    await job.get_async()
    print(job.status)

Job aijob-e00… already finished: COMPLETED
